# 하남교산 격자-위험지수 조인

`02._격자_(하남교산).geojson`과 `08_격자_종합위험지수.csv`를 `gid` 기준으로 병합합니다.

**출력:** `epdo_analysis/output/16_하남교산_위험격자.geojson`

In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

## 경로 설정

노트북이 프로젝트 루트(`LH-Data-Analyze/`)에 위치한다고 가정합니다.  
필요 시 아래 경로를 수정하세요.

In [ ]:
BASE_DIR = Path(".").resolve()

INPUT_GRID = BASE_DIR / "data" / "02._격자_(하남교산).geojson"
INPUT_RISK = BASE_DIR / "epdo_analysis" / "output" / "08_격자_종합위험지수.csv"
OUT_PATH   = BASE_DIR / "epdo_analysis" / "output" / "16_하남교산_위험격자.geojson"

print(f"격자  : {'OK' if INPUT_GRID.exists() else 'NOT FOUND'}  {INPUT_GRID}")
print(f"위험도: {'OK' if INPUT_RISK.exists() else 'NOT FOUND'}  {INPUT_RISK}")

## 데이터 로드

In [ ]:
gdf     = gpd.read_file(INPUT_GRID)
risk_df = pd.read_csv(INPUT_RISK, encoding="utf-8-sig", low_memory=False)

if "gid" not in gdf.columns:
    raise ValueError(f"'gid' column not found in {INPUT_GRID}")
if "grid_gid" not in risk_df.columns:
    raise ValueError(f"'grid_gid' column not found in {INPUT_RISK}")

print(f"격자 행 수  : {len(gdf):,}  CRS: {gdf.crs}")
print(f"위험지수 행 수: {len(risk_df):,}")
print(f"위험지수 컬럼: {risk_df.columns.tolist()}")

## 조인 및 좌표계 변환

In [ ]:
gdf["gid"]          = gdf["gid"].astype(str)
risk_df["grid_gid"] = risk_df["grid_gid"].astype(str)

joined = gdf.merge(risk_df, left_on="gid", right_on="grid_gid", how="left")
joined = joined.to_crs("EPSG:4326")

print(f"전체 행 수     : {len(joined):,}")
print(f"위험도 매핑 수 : {joined['grid_gid'].notna().sum():,} / {len(joined):,}")
joined.head()

## 파일 저장

In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
joined.to_file(OUT_PATH, driver="GeoJSON")

print(f"saved: {OUT_PATH}")